# Notebook 02 - Benchmark solver (R2) and calibration

Verifies the equilibrium solver on the no-shock baseline, cross-checks the new
analytical-Jacobian solver (`core_solver.jl`) against the verified numerical one
(`model.jl`), and reproduces the single-sector oil-shock calibration
(welfare GDP vs Hulten) that the paper uses to size the non-linear amplification.

In [2]:
# --- Project setup (robust path resolution) ---
# @__DIR__ resolves to this notebook's directory; we anchor on the package root.
using LinearAlgebra, Statistics, Printf, DelimitedFiles

const NOTEBOOK_DIR = @__DIR__
const REP_DIR = joinpath(NOTEBOOK_DIR, "..")          # bf_replication/
cd(REP_DIR)
include(joinpath(REP_DIR, "src", "BFReplication.jl"))
using .BFReplication
using .BFReplication.DataLoader
using .BFReplication.BFModel
using .BFReplication.InflationAnalysis

const DATA_DIR = joinpath(REP_DIR, "..", "Replication Files", "GDP Simulatin -- 88 Sector")
const RESULTS_DIR = joinpath(REP_DIR, "data", "results")
mkpath(RESULTS_DIR)

println("Project dir : ", REP_DIR)
println("Data dir    : ", DATA_DIR)
println("Results dir : ", RESULTS_DIR)

# --- Load data ---
data = load_bf_data(joinpath(DATA_DIR, "BFdata.csv"); year=1980)
describe_data(data)

Project dir : /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication/notebooks/..
Data dir    : /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication/notebooks/../../Replication Files/GDP Simulatin -- 88 Sector
Results dir : /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication/notebooks/../data/results
B&F Replication Data Summary
Sectors: 76
Base year: 1980

Ω matrix: 76x76 (cost shares)
α (factor shares): mean=0.4954, min=0.0989, max=0.8264
β (consumption shares): sum=1.0000, max sector=0.1416
λ (Domar weights): sum=2.0918, max=0.1551
L (labor allocation): sum=1.0000

Key checks:
  λ ⊙ α ≈ L? max diff = 0.00e+00
  (I - diag(1-α)·Ω)⁻¹' · β ≈ λ? max diff = 0.00e+00
  β sums to 1? sum = 1.0000000000


## Baseline (no shock)

In [3]:
epsilon, theta, sigma = 0.5, 0.001, 0.9
A0 = ones(data.N)
params0 = BFParameters(A0, data.Ω, data.α, data.β, data.L, epsilon, theta, sigma)
sol0 = compute_equilibrium(params0)
@assert sol0.converged
println("baseline: max|p-1| = $(maximum(abs.(sol0.p .- 1)))")
println("nominal GDP = $(sol0.nominal_gdp),  CPI = $(sol0.cpi)")

baseline: max|p-1| = 0.0
nominal GDP = 0.9999999999999993,  CPI = 0.9999999999999978


## Solver cross-check: numerical (`model.jl`) vs analytical (`core_solver.jl`)
The two solvers implement the SAME equations; their equilibria must agree to ~1e-10.
Run this first after any change to `core_solver.jl`.

In [4]:
A_test = ones(data.N); A_test[7] = 0.7
p_num = BFParameters(A_test, data.Ω, data.α, data.β, data.L, epsilon, theta, sigma)
sol_num = compute_equilibrium(p_num)
sol_an  = solve_bf(A_test, data.Ω, data.α, data.β, data.L, epsilon, theta, sigma; jacobian=:analytical)
println("numerical  GDP = $(sol_num.nominal_gdp)")
println("analytical GDP = $(sol_an.C)")
println("max|p_num - p_an| = $(maximum(abs.(sol_num.p .- sol_an.p)))")

numerical  GDP = 0.9481421096723965
analytical GDP = Inf
max|p_num - p_an| = 9.786357900256733e132


## Single-sector oil shock (calibration): welfare GDP vs Hulten

In [5]:
SHOCK = 7; A_val = 0.7
A = ones(data.N); A[SHOCK] = A_val
ps = BFParameters(A, data.Ω, data.α, data.β, data.L, epsilon, 0.0001, sigma)
sol = compute_equilibrium(ps)
GDP_w = gdp_welfare(sol, data.β, sigma)
p0 = BFParameters(ones(data.N), data.Ω, data.α, data.β, data.L, epsilon, 0.0001, sigma)
sol0b = compute_equilibrium(p0)
GDP0_w = gdp_welfare(sol0b, data.β, sigma)
Delta_welfare = log(GDP_w) - log(GDP0_w)
Hulten = data.λ[SHOCK] * log(A_val)
println("welfare Dlog GDP = $(round(Delta_welfare*100, digits=4))%")
println("Hulten  Dlog     = $(round(Hulten*100, digits=4))%")
println("amplification    = $(round(Delta_welfare/Hulten, digits=3))x")

welfare Dlog GDP = -3.548%
Hulten  Dlog     = -2.7681%
amplification    = 1.282x


## Export oil-shock response curve (welfare vs Hulten)

In [6]:
grid_a = range(0.7, 1.3; length=25)
out = zeros(length(grid_a), 3)
for (k, a) in enumerate(grid_a)
    A = ones(data.N); A[SHOCK] = a
    p = BFParameters(A, data.Ω, data.α, data.β, data.L, epsilon, 0.0001, sigma)
    s = compute_equilibrium(p)
    out[k,1] = a
    out[k,2] = log(gdp_welfare(s, data.β, sigma)) - log(GDP0_w)
    out[k,3] = data.λ[SHOCK] * log(a)
end
writedlm(joinpath(RESULTS_DIR, "oil_shock_response.csv"), out, ',')
println("wrote oil_shock_response.csv")

wrote oil_shock_response.csv
